# Chapter 40: Neighbours

Checks [chapter 40](../../book/40-neighbours.md). The models are small and exact, NumPy in float64, seeded. The truth for
nearest neighbours is a random sum of sines on the plane with a known slope limit (the sum of `amplitude * |frequency vector|`),
so the hypothesis of `abs_sub_nearest_le` holds by construction. k-means is Lloyd's method on random points, written so that the
two halves of a step can be costed separately. The contrastive claim uses random vectors of length one.
These are measurements, not proofs: the proofs are in `Neighbours.lean`. Run every cell, top to bottom. A failed check stops the notebook.

In [1]:
import sys
sys.path.append("../part1")

import numpy as np
from common import use_style, save_figure, report, NEAREST, VERTICAL, BOUND, MODEL, BEFORE
import matplotlib.pyplot as plt

use_style()
SEED = 40
TOLERANCE = 1e-9


def make_truth(rng, term_count=3):
    """A random sum of sines on the plane, and its slope limit: sum of amplitude * length of frequency vector."""
    amplitudes = rng.uniform(0.5, 1.5, term_count)
    frequency_vectors = rng.normal(size=(term_count, 2))
    phases = rng.uniform(0, 2 * np.pi, term_count)

    def truth(points):
        angles = points @ frequency_vectors.T + phases
        return (amplitudes * np.sin(angles)).sum(axis=-1)

    slope_limit = float((amplitudes * np.linalg.norm(frequency_vectors, axis=1)).sum())
    return truth, slope_limit


def get_distances(queries, inputs):
    return np.linalg.norm(queries[:, None, :] - inputs[None, :, :], axis=-1)


def get_cost(points, centres, labels):
    return float(((points - centres[labels]) ** 2).sum())


def get_means(points, labels, cluster_count, old_centres):
    centres = old_centres.copy()
    for cluster in range(cluster_count):
        members = points[labels == cluster]
        if len(members):
            centres[cluster] = members.mean(axis=0)
    return centres


def get_softmax(scores):
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / exps.sum(axis=-1, keepdims=True)

## Claim 0: the three worked examples of the chapter

Predicted: the nearest-neighbour error is 2 and the bound is 2; the k-means costs are 134, 14, 4; the softmax share is
`e / (e + 1)` for both forms, about 0.731. A different number would count against the chapter's arithmetic.

In [2]:
# nearest neighbours: truth 2x, inputs 1, 4, 6, query 3
inputs_by_hand = np.array([1.0, 4.0, 6.0])
nearest_by_hand = inputs_by_hand[np.abs(inputs_by_hand - 3.0).argmin()]
copy_error_by_hand = abs(2 * 3.0 - 2 * nearest_by_hand)
bound_by_hand = 2.0 * abs(3.0 - nearest_by_hand)
report("nearest neighbours by hand [worked example, from the formula]: error equals the bound 2 (a line meets it)",
       nearest_by_hand == 4.0 and copy_error_by_hand == 2.0 and bound_by_hand == 2.0,
       f"nearest {nearest_by_hand}, error {copy_error_by_hand}, bound {bound_by_hand}")

# k-means: points 0, 2, 7, 9; centres 0 and 10; 7 and 9 start with centre 0
points_by_hand = np.array([[0.0], [2.0], [7.0], [9.0]])
centres_by_hand = np.array([[0.0], [10.0]])
start_labels = np.array([0, 0, 0, 0])
cost_start = get_cost(points_by_hand, centres_by_hand, start_labels)
nearest_labels = get_distances(points_by_hand, centres_by_hand).argmin(axis=1)
cost_reassigned = get_cost(points_by_hand, centres_by_hand, nearest_labels)
means_by_hand = get_means(points_by_hand, nearest_labels, 2, centres_by_hand)
cost_moved = get_cost(points_by_hand, means_by_hand, nearest_labels)
report("k-means by hand [worked example, from the formula]: costs 134, 14, 4", (cost_start, cost_reassigned, cost_moved) == (134.0, 14.0, 4.0),
       f"{cost_start:g}, {cost_reassigned:g}, {cost_moved:g}")

# contrastive: query (1, 0), keys (1, 0) and (0, 1)
query_by_hand = np.array([1.0, 0.0])
keys_by_hand = np.array([[1.0, 0.0], [0.0, 1.0]])
dots = keys_by_hand @ query_by_hand
squared_distances = ((keys_by_hand - query_by_hand) ** 2).sum(axis=1)
share_dot = get_softmax(dots)[0]
share_distance = get_softmax(-squared_distances / 2)[0]
report("contrastive by hand [worked example, from the formula]: both shares are e / (e + 1) = 0.7311",
       np.allclose([share_dot, share_distance], np.e / (np.e + 1), atol=1e-12),
       f"dot {share_dot:.6f}, distance {share_distance:.6f}")

[PASS] nearest neighbours by hand [worked example, from the formula]: error equals the bound 2 (a line meets it)  (nearest 4.0, error 2.0, bound 2.0)
[PASS] k-means by hand [worked example, from the formula]: costs 134, 14, 4  (134, 14, 4)
[PASS] contrastive by hand [worked example, from the formula]: both shares are e / (e + 1) = 0.7311  (dot 0.731059, distance 0.731059)


True

## Claim 1: copying the nearest label is wrong by at most slope times distance (`abs_sub_nearest_le`)

Draw 30 random truths on the plane (each with its own slope limit), 12 training inputs in `[-3, 3]^2` and 400 query points per truth.
Predicted: for every query, `|truth(query) - truth(nearest)| <= slope_limit * dist(query, nearest)`, and also
`<= slope_limit * dist(query, other)` for every other training input. One violation counts against the claim.
The vertical error here is the copy error; the Euclidean distance is the distance in the input plane, so the two are reported together.

Control: break the hypothesis by using a slope limit of half the true one. Predicted: the bound then fails for some queries
(a bound that cannot fail would say nothing). I report the fraction that fail and the largest overshoot.

In [3]:
rng = np.random.default_rng(SEED)
TRUTH_COUNT, INPUT_COUNT, QUERY_COUNT = 30, 12, 400
worst_ratio = 0.0
any_other_ok = True
control_fail_count = 0
total_queries = 0
largest_overshoot = 0.0
for _ in range(TRUTH_COUNT):
    truth, slope_limit = make_truth(rng)
    inputs = rng.uniform(-3, 3, (INPUT_COUNT, 2))
    queries = rng.uniform(-3, 3, (QUERY_COUNT, 2))
    distances = get_distances(queries, inputs)
    nearest = distances.argmin(axis=1)
    nearest_distance = distances.min(axis=1)
    copy_error = np.abs(truth(queries) - truth(inputs[nearest]))
    worst_ratio = max(worst_ratio, float((copy_error / np.maximum(slope_limit * nearest_distance, 1e-300)).max()))
    any_other_ok &= bool((copy_error[:, None] <= slope_limit * distances + TOLERANCE).all())
    # control: half the true slope limit
    control_bound = 0.5 * slope_limit * nearest_distance
    control_fail_count += int((copy_error > control_bound + TOLERANCE).sum())
    largest_overshoot = max(largest_overshoot, float((copy_error / np.maximum(control_bound, 1e-300)).max()))
    total_queries += QUERY_COUNT
report("copy error <= slope_limit * nearest distance for all queries",
       worst_ratio <= 1 + TOLERANCE, f"{TRUTH_COUNT} truths, {INPUT_COUNT} inputs each, {total_queries} queries, largest error / bound {worst_ratio:.4f}")
report("copy error <= slope_limit * distance to every other training input", any_other_ok)
fail_fraction = control_fail_count / total_queries
report("control: with half the slope limit the bound fails", control_fail_count > 0,
       f"{control_fail_count} of {total_queries} queries fail ({fail_fraction:.1%}), largest error / half-bound {largest_overshoot:.2f}")

[PASS] copy error <= slope_limit * nearest distance for all queries  (30 truths, 12 inputs each, 12000 queries, largest error / bound 0.9456)
[PASS] copy error <= slope_limit * distance to every other training input
[PASS] control: with half the slope limit the bound fails  (1020 of 12000 queries fail (8.5%), largest error / half-bound 1.89)


True

## Claim 2: more training inputs, a smaller bound

The chapter says that to halve the bound you halve the distance to the nearest example. Predicted: with the same truth and
queries, the *bound* `slope_limit * nearest distance` falls as inputs are added, and the measured error stays below it at every
count. The mean nearest distance in the plane should fall roughly like one over the square root of the count, so quadrupling the
inputs should roughly halve it; I report the measured ratio and do not assert an exact value.

In [4]:
rng = np.random.default_rng(SEED + 1)
truth, slope_limit = make_truth(rng)
queries = rng.uniform(-3, 3, (2000, 2))
rows = []
for count in (10, 40, 160, 640):
    inputs = rng.uniform(-3, 3, (count, 2))
    distances = get_distances(queries, inputs)
    nearest_distance = distances.min(axis=1)
    copy_error = np.abs(truth(queries) - truth(inputs[distances.argmin(axis=1)]))
    rows.append((count, nearest_distance.mean(), copy_error.mean(), slope_limit * nearest_distance.mean()))
    assert (copy_error <= slope_limit * nearest_distance + TOLERANCE).all()
for count, mean_distance, mean_error, mean_bound in rows:
    print(f"inputs {count:4d}: mean nearest distance {mean_distance:.4f}, mean copy error {mean_error:.4f}, mean bound {mean_bound:.4f}")
print("mean bound / mean copy error: " + ", ".join(f"{row[3] / row[2]:.2f}" for row in rows) + "  (the bound is from the formula: slope limit x measured distance)")
ratio_per_quadruple = [rows[index + 1][1] / rows[index][1] for index in range(len(rows) - 1)]
report("the bound falls as inputs are added, and the error stays under it",
       all(rows[index + 1][3] < rows[index][3] for index in range(3)) and all(row[2] <= row[3] + TOLERANCE for row in rows),
       "mean distance ratios per fourfold increase: " + ", ".join(f"{ratio:.3f}" for ratio in ratio_per_quadruple))

inputs   10: mean nearest distance 0.9903, mean copy error 1.0421, mean bound 4.5500
inputs   40: mean nearest distance 0.4298, mean copy error 0.4371, mean bound 1.9746
inputs  160: mean nearest distance 0.2399, mean copy error 0.2512, mean bound 1.1022
inputs  640: mean nearest distance 0.1202, mean copy error 0.1300, mean bound 0.5522
mean bound / mean copy error: 4.37, 4.52, 4.39, 4.25  (the bound is from the formula: slope limit x measured distance)
[PASS] the bound falls as inputs are added, and the error stays under it  (mean distance ratios per fourfold increase: 0.434, 0.558, 0.501)


True

## Claim 3: a k-means step never raises its cost (`sum_sq_reassign_le`, `sum_sq_mean_le`)

Both halves are tested apart.

- Second half: for random clusters, the mean costs no more than any other centre. Predicted: 0 violations over 5000 random
  clusters and random rival centres.
- Whole method: 200 random point sets (60 points in the plane, 5 clusters), started from a random assignment. Each round is
  *assign*, then *move to the mean*. Predicted: the cost sequence is non-increasing at every half step. A single rise counts against the claim.

Control: break the second half by moving each centre to the mean plus an offset of length 1 in a random direction. Predicted:
the cost can rise at some half step. I report how often.

In [5]:
rng = np.random.default_rng(SEED + 2)

# second half alone
violations, strict_gaps = 0, []
for _ in range(5000):
    cluster = rng.normal(size=(rng.integers(2, 12), 2)) * rng.uniform(0.5, 3)
    mean = cluster.mean(axis=0)
    rival = mean + rng.normal(size=2) * rng.uniform(0.01, 2)
    mean_cost = ((cluster - mean) ** 2).sum()
    rival_cost = ((cluster - rival) ** 2).sum()
    violations += int(mean_cost > rival_cost + TOLERANCE)
    strict_gaps.append(rival_cost - mean_cost)
report("mean cost <= rival centre cost, 5000 random clusters", violations == 0,
       f"{violations} violations, smallest gap {min(strict_gaps):.2e}")

POINT_COUNT, CLUSTER_COUNT, ROUNDS, SET_COUNT = 60, 5, 12, 200


def run_lloyd(points, labels, rng, offset_length=0.0):
    """Cost after every half step, starting from an assignment. A positive offset_length breaks the move-to-the-mean half."""
    centres = get_means(points, labels, CLUSTER_COUNT, points[:CLUSTER_COUNT].copy())
    costs = [get_cost(points, centres, labels)]
    for _ in range(ROUNDS):
        labels = get_distances(points, centres).argmin(axis=1)
        costs.append(get_cost(points, centres, labels))
        centres = get_means(points, labels, CLUSTER_COUNT, centres)
        if offset_length > 0:
            directions = rng.normal(size=centres.shape)
            centres = centres + offset_length * directions / np.linalg.norm(directions, axis=1, keepdims=True)
        costs.append(get_cost(points, centres, labels))
    return np.array(costs)


lloyd_costs, control_costs = [], []
for _ in range(SET_COUNT):
    points = np.concatenate([rng.normal(size=(POINT_COUNT // CLUSTER_COUNT, 2)) * 0.6 + rng.uniform(-4, 4, 2)
                             for _ in range(CLUSTER_COUNT)])
    labels = rng.integers(0, CLUSTER_COUNT, POINT_COUNT)
    lloyd_costs.append(run_lloyd(points, labels, rng))
    control_costs.append(run_lloyd(points, labels, rng, offset_length=1.0))
lloyd_costs, control_costs = np.array(lloyd_costs), np.array(control_costs)
largest_rise = float(np.diff(lloyd_costs, axis=1).max())
report("k-means cost is non-increasing at every half step", largest_rise <= TOLERANCE,
       f"{SET_COUNT} point sets of {POINT_COUNT} points in {CLUSTER_COUNT} clusters x {2 * ROUNDS} half steps, largest rise {largest_rise:.2e}, "
       f"mean cost {lloyd_costs[:, 0].mean():.2f} -> {lloyd_costs[:, -1].mean():.2f}")
rise_counts = (np.diff(control_costs, axis=1) > TOLERANCE).sum(axis=1)
report("control: with the centre moved off the mean, the cost rises in some half steps",
       rise_counts.sum() > 0,
       f"{int((rise_counts > 0).sum())} of {SET_COUNT} runs rise at least once, largest single rise {float(np.diff(control_costs, axis=1).max()):.2f}")

[PASS] mean cost <= rival centre cost, 5000 random clusters  (0 violations, smallest gap 2.76e-05)


[PASS] k-means cost is non-increasing at every half step  (200 point sets of 60 points in 5 clusters x 24 half steps, largest rise 0.00e+00, mean cost 525.96 -> 42.08)
[PASS] control: with the centre moved off the mean, the cost rises in some half steps  (200 of 200 runs rise at least once, largest single rise 55.01)


True

## Claim 4: a softmax over cosine similarity is a softmax over minus squared distance (`softmax_cosine_eq_softmax_distance`)

Draw 500 random queries and 8 random keys, all of length one, at temperatures from 0.05 to 20. Predicted: the two sets
of shares agree to rounding error (below `1e-12`). Also `dot = 1 - squared distance / 2` to the same accuracy.

Control: let the keys have lengths between 0.5 and 2. Predicted: the dot product and minus half the squared distance then differ
by an amount that depends on the key, so the shares differ. I report the largest difference.

The chapter also says a low temperature is a hard nearest-neighbour choice and a high temperature a broad average. Predicted:
at temperature 0.05 the top share goes to the nearest key in over 99 percent of queries, and at temperature 20 the mean top share is
within 0.05 of `1 / 8`.

In [6]:
rng = np.random.default_rng(SEED + 3)
KEY_COUNT = 8


def make_unit(vectors):
    return vectors / np.linalg.norm(vectors, axis=-1, keepdims=True)


queries = make_unit(rng.normal(size=(500, 3)))
keys = make_unit(rng.normal(size=(500, KEY_COUNT, 3)))
dots = np.einsum("qd,qkd->qk", queries, keys)
squared_distances = ((queries[:, None, :] - keys) ** 2).sum(axis=-1)
report("dot product = 1 - squared distance / 2 for unit vectors", np.abs(dots - (1 - squared_distances / 2)).max() < 1e-12,
       f"largest difference {np.abs(dots - (1 - squared_distances / 2)).max():.1e}")
worst_gap, summary = 0.0, []
for temperature in (0.05, 0.2, 1.0, 5.0, 20.0):
    cosine_shares = get_softmax(dots / temperature)
    distance_shares = get_softmax(-squared_distances / (2 * temperature))
    gap = float(np.abs(cosine_shares - distance_shares).max())
    worst_gap = max(worst_gap, gap)
    nearest_gets_top = float((cosine_shares.argmax(axis=1) == squared_distances.argmin(axis=1)).mean())
    summary.append((temperature, gap, nearest_gets_top, float(cosine_shares.max(axis=1).mean())))
report("cosine shares equal distance shares at every temperature", worst_gap < 1e-12, f"largest share difference {worst_gap:.1e}")
for temperature, gap, nearest_gets_top, mean_top_share in summary:
    print(f"temperature {temperature:5.2f}: nearest key has top share in {nearest_gets_top:.1%} of queries, mean top share {mean_top_share:.3f}")
report("low temperature is a hard nearest choice, high temperature a broad average",
       summary[0][2] > 0.99 and abs(summary[-1][3] - 1 / KEY_COUNT) < 0.05,
       f"nearest has top share at 0.05 in {summary[0][2]:.3f} of queries; mean top share at 20 is {summary[-1][3]:.3f} against 1/{KEY_COUNT} = {1 / KEY_COUNT:.3f}")

# control: keys not of length one
loose_keys = keys * rng.uniform(0.5, 2.0, (500, KEY_COUNT, 1))
loose_dots = np.einsum("qd,qkd->qk", queries, loose_keys)
loose_squared = ((queries[:, None, :] - loose_keys) ** 2).sum(axis=-1)
loose_gap = float(np.abs(get_softmax(loose_dots) - get_softmax(-loose_squared / 2)).max())
report("control: keys not of length one break the equality", loose_gap > 1e-3, f"key lengths drawn from 0.5 to 2, largest share difference {loose_gap:.3f}")

[PASS] dot product = 1 - squared distance / 2 for unit vectors  (largest difference 6.7e-16)
[PASS] cosine shares equal distance shares at every temperature  (largest share difference 1.8e-15)
temperature  0.05: nearest key has top share in 100.0% of queries, mean top share 0.868
temperature  0.20: nearest key has top share in 100.0% of queries, mean top share 0.599
temperature  1.00: nearest key has top share in 100.0% of queries, mean top share 0.237
temperature  5.00: nearest key has top share in 100.0% of queries, mean top share 0.145
temperature 20.00: nearest key has top share in 100.0% of queries, mean top share 0.130
[PASS] low temperature is a hard nearest choice, high temperature a broad average  (nearest has top share at 0.05 in 1.000 of queries; mean top share at 20 is 0.130 against 1/8 = 0.125)
[PASS] control: keys not of length one break the equality  (key lengths drawn from 0.5 to 2, largest share difference 0.292)


True

## Figure 1: where the nearest neighbour sits, and how wrong it can be

Left: the nearest-neighbour regions of 8 training inputs in the plane (each region is the set of points whose nearest training
input is that one), with one query and the input whose label it copies. Right: for 400 queries on the same truth, the copy error
against the distance to the nearest training input (**measured**), the bound `slope_limit * distance` (**proved**, a line through
the origin), and the control with half the slope limit, where points cross the line.

In [7]:
rng = np.random.default_rng(SEED + 4)
truth, slope_limit = make_truth(rng)
inputs = rng.uniform(-3, 3, (8, 2))
grid_axis = np.linspace(-3.5, 3.5, 400)
grid_x, grid_y = np.meshgrid(grid_axis, grid_axis)
grid = np.stack([grid_x.ravel(), grid_y.ravel()], axis=1)
region = get_distances(grid, inputs).argmin(axis=1).reshape(grid_x.shape)
query = np.array([0.3, 0.4])
query_nearest = get_distances(query[None, :], inputs).argmin()
queries = rng.uniform(-3, 3, (400, 2))
distances = get_distances(queries, inputs)
nearest_distance = distances.min(axis=1)
copy_error = np.abs(truth(queries) - truth(inputs[distances.argmin(axis=1)]))
assert (copy_error <= slope_limit * nearest_distance + TOLERANCE).all()
half_fail = int((copy_error > 0.5 * slope_limit * nearest_distance + TOLERANCE).sum())

figure, (left, right) = plt.subplots(1, 2, figsize=(7, 3.2))
left.pcolormesh(grid_x, grid_y, region, cmap="Pastel1", shading="auto", alpha=0.8)
left.scatter(inputs[:, 0], inputs[:, 1], color="black", s=22, label="training inputs")
left.scatter(*query, color=VERTICAL, marker="*", s=120, label="query", zorder=3)
left.plot([query[0], inputs[query_nearest, 0]], [query[1], inputs[query_nearest, 1]], color=NEAREST, linewidth=1.8,
          label="to its nearest input", zorder=2)
left.set_xlabel("input, first coordinate")
left.set_ylabel("input, second coordinate")
left.set_title("Nearest-neighbour regions (colours)")
left.set_aspect("equal")
left.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=6)
right.scatter(nearest_distance, copy_error, s=8, color=VERTICAL, label="measured copy error")
reach = np.linspace(0, nearest_distance.max(), 50)
right.plot(reach, slope_limit * reach, color=BOUND, label="proved bound: slope limit x distance")
right.plot(reach, 0.5 * slope_limit * reach, color=BEFORE, linestyle="--", label="control: half the slope limit")
right.set_xlabel("distance from query to nearest training input")
right.set_ylabel("copy error (vertical, in the label)")
right.set_title("Copy error under its bound")
right.legend(fontsize=6)
figure.tight_layout()
save_figure(figure, "40-nearest-neighbour-bound")
plt.close(figure)
print(f"{half_fail} of {len(queries)} queries cross the half-slope control line")

wrote book/figures/40-nearest-neighbour-bound.png
54 of 400 queries cross the half-slope control line


## Figure 2: the k-means cost against rounds

The mean cost over the 200 point sets of Claim 3, after each half step, for Lloyd's method and for the control whose centre is
moved off the mean. Lloyd's curve never rises; the control's does.

In [8]:
half_steps = np.arange(lloyd_costs.shape[1])
figure, axis = plt.subplots(figsize=(5.5, 3.2))
for costs, colour, label in ((lloyd_costs, MODEL, "assign, then move to the mean"),
                             (control_costs, BEFORE, "control: move to the mean plus an offset of 1")):
    axis.plot(half_steps, costs.mean(axis=0), color=colour, label=label)
    axis.fill_between(half_steps, np.percentile(costs, 10, axis=0), np.percentile(costs, 90, axis=0), color=colour, alpha=0.2)
axis.set_xlabel("half steps (assign, move, assign, move, ...)")
axis.set_ylabel("cost: sum of squared distances")
axis.set_title("k-means cost, mean over 200 point sets (band: 10th to 90th percentile)")
axis.legend(fontsize=6)
figure.tight_layout()
save_figure(figure, "40-kmeans-cost-against-rounds")
plt.close(figure)
print(f"Lloyd mean cost {lloyd_costs[:, 0].mean():.2f} -> {lloyd_costs[:, -1].mean():.2f}; control mean cost {control_costs[:, 0].mean():.2f} -> {control_costs[:, -1].mean():.2f}")

wrote book/figures/40-kmeans-cost-against-rounds.png
Lloyd mean cost 525.96 -> 42.08; control mean cost 525.96 -> 106.28
